#1: Imports and Path Setup

In [1]:
import os
import sys
import numpy as np
import tensorflow as tf

In [2]:
PROJECT_ROOT = '/content/drive/MyDrive/Transformer_Mine'

In [3]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/Transformer_Mine

Mounted at /content/drive
/content/drive/MyDrive/Transformer_Mine


In [4]:
from multi_head_attention import MultiHeadAttention
from feed_forward import FeedForwardNetwork
from positional_encoding import PositionalEncoding
from data_pipeline import create_padding_mask

print("TensorFlow:", tf.__version__)
print("Project root:", PROJECT_ROOT)

TensorFlow: 2.20.0
Project root: /content/drive/MyDrive/Transformer_Mine


# 2: Mathematical Foundation

From the paper (Section 3.1):

Each encoder layer has two sub-layers:

1. Multi-Head Self-Attention
2. Position-wise Feed-Forward Network

Each sub-layer is wrapped with:

- A residual (skip) connection
- Layer Normalization

Expressed as:

$$
\text{output} = \text{LayerNorm}(x + \text{SubLayer}(x))
$$

This is **POST-LN** (original paper architecture):

$$
x \rightarrow \text{SubLayer}(x) \rightarrow \text{Add}(x, \text{SubLayer}(x)) \rightarrow \text{LayerNorm}
$$

**NOT Pre-LN** (modern variant):

$$
x \rightarrow \text{LayerNorm}(x) \rightarrow \text{SubLayer} \rightarrow \text{Add}(x, \text{SubLayer}(\text{LN}(x)))
$$

We follow the original paper — **POST-LN**.

Full encoder layer:

```text
# Sub-layer 1: self-attention
attn_out = MultiHeadAttention(x, x, x, mask)
attn_out = Dropout(attn_out)
x        = LayerNorm(x + attn_out)

# Sub-layer 2: feed-forward
ffn_out  = FFN(x)
ffn_out  = Dropout(ffn_out)
x        = LayerNorm(x + ffn_out)

# Full Encoder Stack

The encoder follows this architecture:

**src → Embedding → Positional Encoding → [N × EncoderLayer]**

- **N = 6** for Transformer Base.

### Residual Connection

Residual connections allow gradients to flow directly to earlier layers, helping prevent vanishing gradients in deep networks.

### Layer Normalization

Layer Normalization normalizes across the feature dimension (`d_model`) for each position independently. This helps stabilize training.

In [5]:
print("Encoder architecture (Transformer Base):")
print()
print("  src_ids")
print("     |")
print("  Embedding (vocab_size, d_model=512)")
print("     |")
print("  PositionalEncoding (dropout=0.1)")
print("     |")
print("  [EncoderLayer x 6]")
print("     |  - MultiHeadAttention (8 heads)")
print("     |  - Add + LayerNorm")
print("     |  - FeedForwardNetwork (d_ff=2048)")
print("     |  - Add + LayerNorm")
print("     |")
print("  encoder output: (batch, src_len, 512)")

Encoder architecture (Transformer Base):

  src_ids
     |
  Embedding (vocab_size, d_model=512)
     |
  PositionalEncoding (dropout=0.1)
     |
  [EncoderLayer x 6]
     |  - MultiHeadAttention (8 heads)
     |  - Add + LayerNorm
     |  - FeedForwardNetwork (d_ff=2048)
     |  - Add + LayerNorm
     |
  encoder output: (batch, src_len, 512)


# 3: EncoderLayer

One encoder layer consists of two sub-layers.

Each sub-layer uses:
- A residual connection
- Layer normalization

The implementation follows the **Post-LN** architecture described in the original Transformer paper.

In [25]:
class EncoderLayer(tf.keras.layers.Layer):
    """
    Single encoder layer.

    Architecture (POST-LN, original paper):
        x = LayerNorm(x + MultiHeadAttention(x, x, x, mask))
        x = LayerNorm(x + FFN(x))

    Args:
        d_model : model dimension (e.g. 512)
        num_heads : number of attention heads (e.g. 8)
        d_ff : feed-forward inner dimension (e.g. 2048)
        dropout : dropout rate (e.g. 0.1)
    """

    def __init__(self, d_model, num_heads, d_ff, dropout=0.1, **kwargs):
        super().__init__(**kwargs)

        self.d_model   = d_model
        self.num_heads = num_heads
        self.d_ff      = d_ff
        self.dropout_rate = dropout
        self.supports_masking = True

        # Sub-layer 1: multi-head self-attention
        self.self_attention = MultiHeadAttention(
            d_model=d_model,
            num_heads=num_heads,
            name='encoder_self_attention',
        )

        # Sub-layer 2: feed-forward network
        self.ffn = FeedForwardNetwork(
            d_model=d_model,
            d_ff=d_ff,
            dropout=dropout,
            name='encoder_ffn',
        )

        # Layer normalization after each sub-layer
        # epsilon=1e-6
        self.norm_1 = tf.keras.layers.LayerNormalization(
            epsilon=1e-6,
            name='encoder_norm_1',
        )
        self.norm_2 = tf.keras.layers.LayerNormalization(
            epsilon=1e-6,
            name='encoder_norm_2',
        )

        # Dropout applied to sub-layer outputs before residual add
        self.dropout_1 = tf.keras.layers.Dropout(dropout)
        self.dropout_2 = tf.keras.layers.Dropout(dropout)

    def call(self, x, mask=None, training=False):
        """
        Forward pass of one encoder layer.

        Args:
            x        : (batch, seq_len, d_model)
            mask     : padding mask (batch, 1, 1, seq_len)
            training : controls dropout

        Returns:
            x        : (batch, seq_len, d_model)
            attn_weights : (batch, num_heads, seq_len, seq_len)
        """
        # Sub-layer 1: multi-head self-attention
        # In encoder self-attention: Q = K = V = x
        attn_out, attn_weights = self.self_attention(
            x, x, x, mask=mask, training=training
        )

        attn_out = self.dropout_1(attn_out, training=training)

        # Residual connection + layer normalization
        x = self.norm_1(x + attn_out)

        # Sub-layer 2: feed-forward network
        ffn_out = self.ffn(x, training=training)

        ffn_out = self.dropout_2(ffn_out, training=training)

        # Residual connection + layer normalization (POST-LN)
        x = self.norm_2(x + ffn_out)

        return x, attn_weights

    def get_config(self):
        config = super().get_config()

        config.update({
            'd_model' : self.d_model,
            'num_heads' : self.num_heads,
            'd_ff' : self.d_ff,
            'dropout' : self.dropout_rate,
        })

        return config




In [26]:
print("Sub-layer 1: MultiHeadAttention -> Dropout -> Add -> LayerNorm")
print("Sub-layer 2: FeedForwardNetwork -> Dropout -> Add -> LayerNorm")

Sub-layer 1: MultiHeadAttention -> Dropout -> Add -> LayerNorm
Sub-layer 2: FeedForwardNetwork -> Dropout -> Add -> LayerNorm


# 4: Encoder Stack

Stacks **N encoder layers** with an embedding layer and positional encoding at the input.

The embedding is scaled by `sqrt(d_model)` inside `PositionalEncoding.call()`.

Therefore, no separate embedding scaling is required here.

In [34]:
class Encoder(tf.keras.layers.Layer):
    """
    Full encoder stack.

    src_ids -> Embedding -> PositionalEncoding -> [EncoderLayer x N]

    Args:
        num_layers : number of encoder layers (paper: 6)
        d_model : model dimension (paper: 512)
        num_heads : attention heads (paper: 8)
        d_ff : FFN inner dim (paper: 2048)
        vocab_size : source vocabulary size
        max_len : max sequence length for positional encoding
        dropout : dropout rate (paper: 0.1)
    """

    def __init__(self, num_layers, d_model, num_heads, d_ff, vocab_size, max_len = 5000, dropout = 0.1, **kwargs):
        super().__init__(**kwargs)

        self.num_layers = num_layers
        self.d_model = d_model
        self.supports_masking = True

        # Token embedding : Converts integer token IDs to d_model-dimensional vectors
        self.embedding = tf.keras.layers.Embedding(
            input_dim = vocab_size,
            output_dim = d_model,
            name='encoder_embedding',
        )

        # Positional encoding with embedding scaling + dropout
        self.pos_encoding = PositionalEncoding(
            max_len = max_len,
            d_model = d_model,
            dropout = dropout,
            name='encoder_pos_encoding',
        )

        # Stack of N encoder layers
        self.enc_layers = [
            EncoderLayer(
                d_model = d_model,
                num_heads = num_heads,
                d_ff = d_ff,
                dropout = dropout,
                name=f'encoder_layer_{i}',
            )

            for i in range(num_layers)
        ]

    def call(self, src_ids, mask = None, training = False):
        """
        Forward pass of the full encoder.

        Args:
            src_ids : integer token IDs (batch, src_len)
            mask : padding mask (batch, 1, 1, src_len)
            training : controls dropout

        Returns:
            x : encoder output (batch, src_len, d_model)
            attention_weights : dict of attn weights per layer
        """

        # Embed tokens
        x = self.embedding(src_ids)

        # Add positional encoding (also scales by sqrt(d_model))
        x =self.pos_encoding(x,training = training)

        # Pass through N encoder layers
        attention_weights = {}

        for i, layer in enumerate (self.enc_layers):
            x, attn_w = layer(x, mask=mask, training=training)

            attention_weights[f'encoder_layer_{i+1}_attn'] = attn_w

        return x, attention_weights

    def get_config(self):
        config = super().get_config()

        config.update({
            'num_layers' : self.num_layers,
            'd_model' : self.d_model,
        })

        return config

In [35]:
print("src_ids -> Embedding -> PositionalEncoding -> [EncoderLayer x N]")
print("Returns encoder output and attention weights from all layers.")

src_ids -> Embedding -> PositionalEncoding -> [EncoderLayer x N]
Returns encoder output and attention weights from all layers.


#5: Shape Tests

In [36]:
NUM_LAYERS = 6
D_MODEL = 512
NUM_HEADS = 8
D_FF = 2048
VOCAB_SIZE = 8000
MAX_LEN = 100
BATCH = 2
SRC_LEN = 20

In [37]:
encoder = Encoder(
    num_layers = NUM_LAYERS,
    d_model = D_MODEL,
    num_heads = NUM_HEADS,
    d_ff = D_FF,
    vocab_size = VOCAB_SIZE,
    max_len = MAX_LEN,
    dropout = 0.0,
)

In [38]:
tf.random.set_seed(42)

In [39]:
src_ids = tf.constant([
    [5, 12, 8, 33, 7, 2, 0, 0, 0, 0,
     0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    [3, 18, 6, 44, 9, 11, 22, 4, 2, 0,
     0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
], dtype=tf.int32)

In [40]:
pad_mask = create_padding_mask(src_ids)

In [41]:
enc_output, attn_weights = encoder(src_ids, mask=pad_mask, training=False)


/usr/local/lib/python3.13/dist-packages/keras/src/layers/layer.py:982: UserWarning: Layer 'encoder_self_attention' (of type MultiHeadAttention) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(


In [42]:
# Verify output shape
assert enc_output.shape == (BATCH, SRC_LEN, D_MODEL), \
    f"Encoder output shape wrong: {enc_output.shape}"

print(f"input shape : {src_ids.shape} (batch, src_len) - integer token IDs")

print(f"mask shape : {pad_mask.shape}")

print(f"output shape : {enc_output.shape} (batch, src_len, d_model)")

input shape : (2, 20) (batch, src_len) - integer token IDs
mask shape : (2, 1, 1, 20)
output shape : (2, 20, 512) (batch, src_len, d_model)


In [43]:
# Verify attention weights from all layers
for layer_name, weights in attn_weights.items():
    expected = (BATCH, NUM_HEADS, SRC_LEN, SRC_LEN)

    assert weights.shape == expected, \
        f"{layer_name} weights shape wrong: {weights.shape}"

    print(f"{layer_name}: {weights.shape}")



encoder_layer_1_attn: (2, 8, 20, 20)
encoder_layer_2_attn: (2, 8, 20, 20)
encoder_layer_3_attn: (2, 8, 20, 20)
encoder_layer_4_attn: (2, 8, 20, 20)
encoder_layer_5_attn: (2, 8, 20, 20)
encoder_layer_6_attn: (2, 8, 20, 20)


# 6: Parameter Count

Counts and breaks down all encoder parameters.

Compares the calculated parameter count against the expected values from the paper.

In [44]:
# Force weight creation
_ = encoder(
    tf.zeros((1, 10), dtype=tf.int32),
    training=False
)

In [47]:
# Embedding parameters
embed_params = VOCAB_SIZE * D_MODEL
print(f"\nEmbedding layer:")
print(f"vocab_size x d_model = {VOCAB_SIZE} x {D_MODEL} = {embed_params:,}")


Embedding layer:
vocab_size x d_model = 8000 x 512 = 4,096,000


In [48]:
# Per encoder layer
mha_params_per_layer = 4 * D_MODEL * D_MODEL
ffn_params_per_layer = 2 * D_MODEL * D_FF + D_FF + D_MODEL
ln_params_per_layer  = 2 * 2 * D_MODEL   # 2 LN layers, each has gamma+beta
total_per_layer = mha_params_per_layer + ffn_params_per_layer + ln_params_per_layer

In [49]:
print(f"\nPer encoder layer:")
print(f"MultiHeadAttention : 4 x {D_MODEL}^2 = {mha_params_per_layer:,}")

print(f"FeedForwardNetwork : 2 x {D_MODEL} x {D_FF} + {D_FF} + {D_MODEL} = {ffn_params_per_layer:,}")

print(f"LayerNorm (x2): 2 x 2 x {D_MODEL}        = {ln_params_per_layer:,}")

print(f"  Total per layer:= {total_per_layer:,}")

total_layers = NUM_LAYERS * total_per_layer
total_encoder = embed_params + total_layers

print(f"\nTotal ({NUM_LAYERS} layers):")
print(f"Embedding : {embed_params:,}")
print(f"{NUM_LAYERS} x EncoderLayer   : {total_layers:,}")
print(f"Grand total: {total_encoder:,}")

# Count actual trainable parameters
actual_total = sum(tf.size(w).numpy() for w in encoder.trainable_variables)

print(f"\nActual trainable parameters: {actual_total:,}")

print(f"Non-trainable parameters : "
      f"{sum(tf.size(w).numpy() for w in encoder.non_trainable_variables):,}")


Per encoder layer:
MultiHeadAttention : 4 x 512^2 = 1,048,576
FeedForwardNetwork : 2 x 512 x 2048 + 2048 + 512 = 2,099,712
LayerNorm (x2): 2 x 2 x 512        = 2,048
  Total per layer:= 3,150,336

Total (6 layers):
Embedding : 4,096,000
6 x EncoderLayer   : 18,902,016
Grand total: 22,998,016

Actual trainable parameters: 22,998,016
Non-trainable parameters : 0


# Residual Connection and Layer Normalization

Explicitly verifies that:

1. The residual connection is active — the output differs from the sub-layer output alone.
2. Layer Normalization normalizes correctly — mean ≈ 0 and standard deviation ≈ 1 per position.
3. The **POST-LN** order is correct: **Add → Norm**.

In [50]:
enc_layer_test = EncoderLayer(
    d_model=D_MODEL, num_heads=NUM_HEADS,
    d_ff=D_FF, dropout=0.0,
)

In [51]:
tf.random.set_seed(0)

In [52]:
x_test = tf.random.normal((1, 10, D_MODEL))
mask_test = None

output_test, _ = enc_layer_test(x_test, mask=mask_test, training=False)

In [53]:
# Test 1: Output is not identical to input (transformation occurred)
diff = tf.reduce_mean(tf.abs(output_test - x_test)).numpy()

print(f"mean abs diff (output vs input): {diff:.6f}")
assert diff > 1e-3
print("layer is applying a non-trivial transformation - correct")

mean abs diff (output vs input): 0.490959
layer is applying a non-trivial transformation - correct


In [54]:
# Test 2: LayerNorm output has mean ~0 and std ~1 per position

out_np = output_test.numpy()
means = out_np[0].mean(axis=-1)
stds  = out_np[0].std(axis=-1)

print(f"\nTest 2 - LayerNorm output statistics per position:")

print(f"mean across positions: min={means.min():.4f}  max={means.max():.4f}  "
      f"(expected ~0)")

print(f"std across positions: min={stds.min():.4f}  max={stds.max():.4f}  "
      f"(expected ~1)")

assert abs(means).max() < 0.1, \
    f"LayerNorm mean too far from 0: {abs(means).max()}"

assert abs(stds - 1.0).max() < 0.15, \
    f"LayerNorm std too far from 1: {abs(stds-1.0).max()}"

print("LayerNorm statistics correct")


Test 2 - LayerNorm output statistics per position:
mean across positions: min=-0.0000  max=0.0000  (expected ~0)
std across positions: min=1.0000  max=1.0000  (expected ~1)
LayerNorm statistics correct


In [56]:
# Test 3: Residual — output should be closer to input than to zero
dist_to_input = tf.norm(output_test - x_test).numpy()
dist_to_zero  = tf.norm(output_test).numpy()

print(f"\nTest 3 - Residual connection keeps output close to input:")
print(f"||output - input|| : {dist_to_input:.4f}")
print(f"||output|| : {dist_to_zero:.4f}")
print("residual connection confirmed active")



Test 3 - Residual connection keeps output close to input:
||output - input|| : 43.9265
||output|| : 71.5541
residual connection confirmed active


# 8: Padding Mask Test

Verifies that **PAD token positions** in the encoder input do not influence the output at non-PAD positions.

This is critical because **PAD tokens should be invisible to the model**.

In [57]:
tf.random.set_seed(5)

In [58]:
seq_a = tf.constant([[10, 20, 30, 40, 50,  0,  0,  0]], dtype=tf.int32)
seq_b = tf.constant([[10, 20, 30, 40, 50, 99, 99, 99]], dtype=tf.int32)

In [59]:
mask_a = create_padding_mask(seq_a)
mask_b = create_padding_mask(seq_b)

In [60]:
enc_test = Encoder(
    num_layers=2, d_model=D_MODEL, num_heads=NUM_HEADS,
    d_ff=D_FF, vocab_size=VOCAB_SIZE, dropout=0.0,
)

In [61]:
out_a, _ = enc_test(seq_a, mask=mask_a, training=False)
out_b, _ = enc_test(seq_b, mask=mask_b, training=False)

In [62]:
# Positions 0-4 should match between seq_a and seq_b
diff_non_pad = tf.reduce_max(
    tf.abs(out_a[0, :5, :] - out_b[0, :5, :])
).numpy()

In [63]:

print(f"Max diff at non-PAD positions (0-4): {diff_non_pad:.6f}")
print("(should be ~0 — PAD positions should not affect non-PAD output)")

if diff_non_pad < 1e-5:
    print("Padding mask is working correctly.")
else:
    print("WARNING: PAD positions are leaking into non-PAD output.")
    print("Check that padding mask is being passed correctly.")



Max diff at non-PAD positions (0-4): 1.717520
(should be ~0 — PAD positions should not affect non-PAD output)
Check that padding mask is being passed correctly.


#9: Gradient Flow Test

In [66]:
tf.random.set_seed(42)

In [67]:
enc_grad = Encoder(
    num_layers=2, d_model=D_MODEL, num_heads=NUM_HEADS,
    d_ff=D_FF, vocab_size=VOCAB_SIZE, dropout=0.0,
)

In [68]:
src_grad = tf.constant(
    [[5, 12, 8, 33, 7, 0, 0, 0],
     [3, 18, 6, 44, 9, 11, 0, 0]],
    dtype=tf.int32,
)

pad_mask_grad = create_padding_mask(src_grad)

In [69]:
with tf.GradientTape() as tape:
    out_g, _ = enc_grad(src_grad, mask=pad_mask_grad, training=True)
    loss_g = tf.reduce_sum(out_g)

grads = tape.gradient(loss_g, enc_grad.trainable_variables)

print(f"\nTotal parameter tensors : {len(enc_grad.trainable_variables)}")

none_grads = [v.name for v, g in zip(enc_grad.trainable_variables, grads)
               if g is None]
zero_grads = [v.name for v, g in zip(enc_grad.trainable_variables, grads)
               if g is not None and tf.norm(g).numpy() == 0]
nan_grads = [v.name for v, g in zip(enc_grad.trainable_variables, grads)
               if g is not None and tf.reduce_any(tf.math.is_nan(g)).numpy()]

print(f"None gradients : {len(none_grads)}")
print(f"Zero gradients : {len(zero_grads)}")
print(f"NaN  gradients : {len(nan_grads)}")

if none_grads:
    print("Parameters with None gradient:")
    for name in none_grads:
        print(f"  {name}")

if nan_grads:
    print("Parameters with NaN gradient:")
    for name in nan_grads:
        print(f"  {name}")

if not none_grads and not nan_grads:
    print("\nGradient flow test passed.")
    print("All parameters receive valid gradients.")
else:
    print("\nGradient flow test FAILED.")

/usr/local/lib/python3.13/dist-packages/keras/src/layers/layer.py:982: UserWarning: Layer 'encoder_self_attention' (of type MultiHeadAttention) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(



Total parameter tensors : 25
None gradients : 0
Zero gradients : 0
NaN  gradients : 0

Gradient flow test passed.
All parameters receive valid gradients.


#10: Save Encoder to Drive

In [70]:
ENCODER_PY = os.path.join(PROJECT_ROOT, 'encoder.py')

In [71]:
code = '''"""
encoder.py - Transformer Encoder

Usage:
    from google.colab import drive
    drive.mount('/content/drive')
    %cd /content/drive/MyDrive/Transformer_Mine
    from encoder import EncoderLayer, Encoder
"""

import tensorflow as tf
from multi_head_attention import MultiHeadAttention
from feed_forward import FeedForwardNetwork
from positional_encoding import PositionalEncoding


class EncoderLayer(tf.keras.layers.Layer):
    """
    Single encoder layer (POST-LN, original paper).

    x = LayerNorm(x + MultiHeadAttention(x, x, x, mask))
    x = LayerNorm(x + FFN(x))
    """

    def __init__(self, d_model, num_heads, d_ff, dropout=0.1, **kwargs):
        super().__init__(**kwargs)
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_ff = d_ff
        self.dropout_rate = dropout
        self.supports_masking = True

        self.self_attention = MultiHeadAttention(d_model, num_heads,
                                                 name="encoder_self_attention")
        self.ffn = FeedForwardNetwork(d_model, d_ff, dropout,
                                                 name="encoder_ffn")
        self.norm_1 = tf.keras.layers.LayerNormalization(epsilon=1e-6,
                                                                  name="encoder_norm_1")
        self.norm_2 = tf.keras.layers.LayerNormalization(epsilon=1e-6,
                                                                  name="encoder_norm_2")
        self.dropout_1 = tf.keras.layers.Dropout(dropout)
        self.dropout_2 = tf.keras.layers.Dropout(dropout)

    def call(self, x, mask=None, training=False):
        attn_out, attn_weights = self.self_attention(x, x, x,
                                                      mask=mask,
                                                      training=training)

        attn_out = self.dropout_1(attn_out, training=training)
        x = self.norm_1(x + attn_out)
        ffn_out = self.ffn(x, training=training)
        ffn_out = self.dropout_2(ffn_out, training=training)
        x = self.norm_2(x + ffn_out)
        return x, attn_weights

    def get_config(self):
        config = super().get_config()
        config.update({
            "d_model": self.d_model,
            "num_heads": self.num_heads,
            "d_ff": self.d_ff,
            "dropout": self.dropout_rate}
        )

        return config


class Encoder(tf.keras.layers.Layer):
    """
    Full encoder stack.
    src_ids -> Embedding -> PositionalEncoding -> [EncoderLayer x N]
    """

    def __init__(self, num_layers, d_model, num_heads, d_ff,
                 vocab_size, max_len=5000, dropout=0.1, **kwargs):
        super().__init__(**kwargs)
        self.num_layers = num_layers
        self.d_model = d_model
        self.supports_masking = True
        self.embedding = tf.keras.layers.Embedding(vocab_size, d_model,
                                                       name="encoder_embedding")
        self.pos_encoding = PositionalEncoding(d_model, max_len, dropout,
                                               name="encoder_pos_encoding")
        self.enc_layers = [
            EncoderLayer(d_model, num_heads, d_ff, dropout,
                         name=f"encoder_layer_{i}")
            for i in range(num_layers)
        ]

    def call(self, src_ids, mask=None, training=False):
        x = self.embedding(src_ids)
        x = self.pos_encoding(x, training=training)
        attention_weights = {}
        for i, layer in enumerate(self.enc_layers):
            x, attn_w = layer(x, mask=mask, training=training)
            attention_weights[f"encoder_layer_{i}"] = attn_w
        return x, attention_weights

    def get_config(self):
        config = super().get_config()
        config.update({"num_layers": self.num_layers, "d_model": self.d_model})
        return config
'''



In [72]:
with open(ENCODER_PY, 'w', encoding='utf-8') as f:
    f.write(code)

In [74]:

import importlib
import encoder
importlib.reload(encoder)
from encoder import Encoder as Encoder_loaded

enc_reload = Encoder_loaded(
    num_layers=6, d_model=512, num_heads=8,
    d_ff=2048, vocab_size=8000, dropout=0.0,
)
src_r = tf.constant([[5, 12, 8, 33, 7, 0, 0, 0]], dtype=tf.int32)
out_r, attn_r = enc_reload(src_r, training=False)

assert out_r.shape == (1, 8, 512)
assert len(attn_r) == 6

print("encoder.py saved and reloaded.")
print(f"path : {ENCODER_PY}")
print(f"output shape : {out_r.shape}")
print(f"attn layers : {len(attn_r)}")

encoder.py saved and reloaded.
path : /content/drive/MyDrive/Transformer_Mine/encoder.py
output shape : (1, 8, 512)
attn layers : 6
